In [1]:
import os
import glob
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, wasserstein_distance, skew, kurtosis

In [2]:
ref_dir="../data/fake_individual_gbm/"
ref_stats_path="../data/fake_individual_gbm/fake_stats_20260320_101045.pkl"
gen_dir="../data/model_generated_gbm/"
gen_stats_path="../data/model_generated_gbm/fake_stats_generated.pkl"

# Statistical Analysis of Toy Dataset

It loads the `stats_dict` from the pickle file and it loads every processed CSV into a dicionary keyed by ticker.

In [3]:
def load_stats_dict(stats_path):
    with open(stats_path, "rb") as f:
        stats_dict = pickle.load(f)
    return stats_dict


def load_processed_universe(data_dir):
    csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    data = {}

    for fp in csv_files:
        name = Path(fp).stem
        # ticker is the first chunk before the first underscore date part
        # for FAKE_0001_1986-01-02_2024-... this keeps FAKE_0001
        parts = name.split("_")
        ticker = "_".join(parts[:2]) if parts[0] == "FAKE" else parts[0]

        df = pd.read_csv(fp)
        data[ticker] = df

    return data

## Destandardizing
We are destandardizing the processed data so that:
* OHCL log-ratios relative to previous close
* volume log-differences

In [4]:
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]


def destandardize_processed_df(df_processed, stats):
    df = df_processed.copy()

    arr_std = df[FEATURE_COLS].to_numpy(dtype=np.float64)
    mean = np.asarray(stats["mean"], dtype=np.float64)
    std = np.asarray(stats["std"], dtype=np.float64)

    arr = arr_std * (std + 1e-8) + mean

    df_out = pd.DataFrame(arr, columns=FEATURE_COLS)
    df_out.insert(0, "Date", pd.to_datetime(df["Date"], format="%d/%m/%Y"))
    return df_out

## Flatten all stock into one table
We generate an aggregate table over all samples as to perform aggregate statistics and time-specific ones.

In [5]:
def make_long_panel(universe_processed, stats_dict):
    rows = []

    for ticker, df_proc in universe_processed.items():
        if ticker not in stats_dict:
            continue

        df_destd = destandardize_processed_df(df_proc, stats_dict[ticker]).copy()
        df_destd["Ticker"] = ticker
        df_destd["Step"] = np.arange(len(df_destd))
        rows.append(df_destd)

    panel = pd.concat(rows, axis=0, ignore_index=True)
    return panel

# Aggregate distribution statistics
* mean
* std
* skewness
* excess kurtosis
* selected quantiles

In [6]:
def summarize_global_distribution(panel):
    summary = {}

    for col in FEATURE_COLS:
        x = panel[col].dropna().to_numpy()

        summary[col] = {
            "mean": np.mean(x),
            "std": np.std(x, ddof=0),
            "skew": skew(x, bias=False),
            "kurtosis_excess": kurtosis(x, fisher=True, bias=False),
            "q01": np.quantile(x, 0.01),
            "q05": np.quantile(x, 0.05),
            "median": np.quantile(x, 0.50),
            "q95": np.quantile(x, 0.95),
            "q99": np.quantile(x, 0.99),
        }

    return pd.DataFrame(summary).T

## Two-sample comparison metrics
For each channel it compares reference and generated distributions using:
* KS statistics
* KS p-value
* Wasserstein-1 distance
* moment differences

In [7]:
def compare_global_distributions(panel_ref, panel_gen):
    rows = []

    for col in FEATURE_COLS:
        x = panel_ref[col].dropna().to_numpy()
        y = panel_gen[col].dropna().to_numpy()

        ks_stat, ks_p = ks_2samp(x, y)
        w1 = wasserstein_distance(x, y)

        rows.append({
            "feature": col,
            "ref_mean": np.mean(x),
            "gen_mean": np.mean(y),
            "ref_std": np.std(x, ddof=0),
            "gen_std": np.std(y, ddof=0),
            "ref_skew": skew(x, bias=False),
            "gen_skew": skew(y, bias=False),
            "ref_kurtosis_excess": kurtosis(x, fisher=True, bias=False),
            "gen_kurtosis_excess": kurtosis(y, fisher=True, bias=False),
            "KS_stat": ks_stat,
            "KS_pvalue": ks_p,
            "Wasserstein_1": w1,
        })

    return pd.DataFrame(rows)

## Pathwise stock-level statistics
For each stock/path it computes:
* mean, variance, skewness, kurtosis of `Close` log-returns
* annualized volatility

This is done to check if aggregating hide a possible path-level mismatch.

In [8]:
def compute_pathwise_stats(universe_processed, stats_dict, T=40.0):
    rows = []

    for ticker, df_proc in universe_processed.items():
        if ticker not in stats_dict:
            continue

        df = destandardize_processed_df(df_proc, stats_dict[ticker])
        r = df["Close"].to_numpy(dtype=np.float64)

        # Pathwise moments on Close log-returns
        mean_r = np.mean(r)
        var_r = np.var(r, ddof=0)
        skew_r = skew(r, bias=False)
        kurt_r = kurtosis(r, fisher=True, bias=False)

        # GBM-style annualized volatility and drift
        # Thesis formulas:
        # nu_hat = sqrt((1/T) * sum r_t^2)
        # mu_hat = (1/T) * sum r_t + 0.5 * nu_hat^2
        nu_hat = np.sqrt(np.sum(r**2) / T)
        mu_hat = np.sum(r) / T + 0.5 * nu_hat**2

        rows.append({
            "Ticker": ticker,
            "mean_close_ret": mean_r,
            "var_close_ret": var_r,
            "skew_close_ret": skew_r,
            "kurt_close_ret_excess": kurt_r,
            "annualized_vol_close": nu_hat,
            "annualized_drift_close": mu_hat,
        })

    return pd.DataFrame(rows)

In [9]:
def compare_pathwise_stats(stats_ref, stats_gen):
    cols = [c for c in stats_ref.columns if c != "Ticker"]
    rows = []

    for col in cols:
        x = stats_ref[col].dropna().to_numpy()
        y = stats_gen[col].dropna().to_numpy()

        ks_stat, ks_p = ks_2samp(x, y)
        w1 = wasserstein_distance(x, y)

        rows.append({
            "metric": col,
            "ref_mean": np.mean(x),
            "gen_mean": np.mean(y),
            "ref_std": np.std(x, ddof=0),
            "gen_std": np.std(y, ddof=0),
            "KS_stat": ks_stat,
            "KS_pvalue": ks_p,
            "Wasserstein_1": w1,
        })

    return pd.DataFrame(rows)

## Step-specific statistics
We are comparing the relative step, which is usually better that comparing calendar dates. For each step across all stocks we compute:
* mean
* std
* 5th percentile
*95th percentile

In [10]:
def stepwise_summary(panel):
    out = (
        panel.groupby("Step")[FEATURE_COLS]
        .agg(["mean", "std", lambda x: np.quantile(x, 0.05), lambda x: np.quantile(x, 0.95)])
    )
    out.columns = [
        f"{col}_{stat if isinstance(stat, str) else ('q05' if i % 4 == 2 else 'q95')}"
        for i, (col, stat) in enumerate(out.columns)
    ]
    out = out.reset_index()
    return out

# Date-specific statistics
Suppose that both dataset use same/similar statistics

In [11]:
def datewise_summary(panel):
    out = (
        panel.groupby("Date")[FEATURE_COLS]
        .agg(["mean", "std", lambda x: np.quantile(x, 0.05), lambda x: np.quantile(x, 0.95)])
    )
    out.columns = [
        f"{col}_{stat if isinstance(stat, str) else ('q05' if i % 4 == 2 else 'q95')}"
        for i, (col, stat) in enumerate(out.columns)
    ]
    out = out.reset_index()
    return out

# Distribution plots
We are plotting as to see:
* wider tails
* shifted mean
* over-dispersion
* asymmetry

In [12]:
def plot_feature_histograms(panel_ref, panel_gen, bins=100):
    fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(10, 3 * len(FEATURE_COLS)))

    for ax, col in zip(axes, FEATURE_COLS):
        x = panel_ref[col].dropna().to_numpy()
        y = panel_gen[col].dropna().to_numpy()

        ax.hist(x, bins=bins, density=True, alpha=0.5, label="Reference")
        ax.hist(y, bins=bins, density=True, alpha=0.5, label="Generated")
        ax.set_title(f"Distribution comparison: {col}")
        ax.legend()

    plt.tight_layout()
    plt.show()

## ECDF plots
It plots empirical CDFs for each feature. This is visually tied to the KS statistic: the KS distance is the maximum gap between these two curves.

In [13]:
def plot_ecdf_comparison(panel_ref, panel_gen):
    fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(10, 3 * len(FEATURE_COLS)))

    for ax, col in zip(axes, FEATURE_COLS):
        x = np.sort(panel_ref[col].dropna().to_numpy())
        y = np.sort(panel_gen[col].dropna().to_numpy())

        fx = np.arange(1, len(x) + 1) / len(x)
        fy = np.arange(1, len(y) + 1) / len(y)

        ax.plot(x, fx, label="Reference")
        ax.plot(y, fy, label="Generated")
        ax.set_title(f"ECDF comparison: {col}")
        ax.legend()

    plt.tight_layout()
    plt.show()

## Pathwise metric plots
We are plotting a stock-level distribution of:
* Annualized drift
* annualized volatility
* mean/variance/skew/kurtosis of returns

In [14]:
def plot_pathwise_metric_histograms(stats_ref, stats_gen):
    metric_cols = [c for c in stats_ref.columns if c != "Ticker"]
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(10, 3 * len(metric_cols)))

    for ax, col in zip(axes, metric_cols):
        x = stats_ref[col].dropna().to_numpy()
        y = stats_gen[col].dropna().to_numpy()

        ax.hist(x, bins=50, density=True, alpha=0.5, label="Reference")
        ax.hist(y, bins=50, density=True, alpha=0.5, label="Generated")
        ax.set_title(f"Pathwise distribution: {col}")
        ax.legend()

    plt.tight_layout()
    plt.show()

## Time-step graph with uncertainty bands

In [15]:
def plot_stepwise_bands(step_ref, step_gen, feature="Close"):
    x_ref = step_ref["Step"].to_numpy()
    x_gen = step_gen["Step"].to_numpy()

    plt.figure(figsize=(12, 5))

    plt.plot(x_ref, step_ref[f"{feature}_mean"], label="Reference mean")
    plt.fill_between(
        x_ref,
        step_ref[f"{feature}_q05"],
        step_ref[f"{feature}_q95"],
        alpha=0.2
    )

    plt.plot(x_gen, step_gen[f"{feature}_mean"], label="Generated mean")
    plt.fill_between(
        x_gen,
        step_gen[f"{feature}_q05"],
        step_gen[f"{feature}_q95"],
        alpha=0.2
    )

    plt.title(f"Stepwise comparison with 5%-95% band: {feature}")
    plt.xlabel("Step")
    plt.ylabel(feature)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Feature correlation comparison

In [16]:
def feature_correlation_matrix(panel):
    return panel[FEATURE_COLS].corr()


def plot_correlation_matrices(panel_ref, panel_gen):
    corr_ref = feature_correlation_matrix(panel_ref)
    corr_gen = feature_correlation_matrix(panel_gen)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im0 = axes[0].imshow(corr_ref, vmin=-1, vmax=1)
    axes[0].set_title("Reference feature correlation")
    axes[0].set_xticks(range(len(FEATURE_COLS)))
    axes[0].set_xticklabels(FEATURE_COLS, rotation=45)
    axes[0].set_yticks(range(len(FEATURE_COLS)))
    axes[0].set_yticklabels(FEATURE_COLS)

    im1 = axes[1].imshow(corr_gen, vmin=-1, vmax=1)
    axes[1].set_title("Generated feature correlation")
    axes[1].set_xticks(range(len(FEATURE_COLS)))
    axes[1].set_xticklabels(FEATURE_COLS, rotation=45)
    axes[1].set_yticks(range(len(FEATURE_COLS)))
    axes[1].set_yticklabels(FEATURE_COLS)

    fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.8)
    plt.tight_layout()
    plt.show()

    return corr_ref, corr_gen

# Run the comparison

In [17]:
def evaluate_universes(
    ref_dir,
    ref_stats_path,
    gen_dir,
    gen_stats_path,
):
    # Load
    ref_universe = load_processed_universe(ref_dir)
    gen_universe = load_processed_universe(gen_dir)

    ref_stats_dict = load_stats_dict(ref_stats_path)
    gen_stats_dict = load_stats_dict(gen_stats_path)

    # Build long panels
    panel_ref = make_long_panel(ref_universe, ref_stats_dict)
    panel_gen = make_long_panel(gen_universe, gen_stats_dict)

    # Global summaries
    global_ref = summarize_global_distribution(panel_ref)
    global_gen = summarize_global_distribution(panel_gen)
    global_cmp = compare_global_distributions(panel_ref, panel_gen)

    # Pathwise summaries
    path_ref = compute_pathwise_stats(ref_universe, ref_stats_dict)
    path_gen = compute_pathwise_stats(gen_universe, gen_stats_dict)
    path_cmp = compare_pathwise_stats(path_ref, path_gen)

    # Time summaries
    step_ref = stepwise_summary(panel_ref)
    step_gen = stepwise_summary(panel_gen)

    # Print
    print("GLOBAL REFERENCE SUMMARY")
    display(global_ref)

    print("GLOBAL GENERATED SUMMARY")
    display(global_gen)

    print("GLOBAL COMPARISON")
    display(global_cmp)

    print("PATHWISE COMPARISON")
    display(path_cmp)

    # Plots
    plot_feature_histograms(panel_ref, panel_gen)
    plot_ecdf_comparison(panel_ref, panel_gen)
    plot_pathwise_metric_histograms(path_ref, path_gen)
    plot_stepwise_bands(step_ref, step_gen, feature="Close")
    plot_stepwise_bands(step_ref, step_gen, feature="Volume")
    plot_correlation_matrices(panel_ref, panel_gen)

    return {
        "panel_ref": panel_ref,
        "panel_gen": panel_gen,
        "global_ref": global_ref,
        "global_gen": global_gen,
        "global_cmp": global_cmp,
        "path_ref": path_ref,
        "path_gen": path_gen,
        "path_cmp": path_cmp,
        "step_ref": step_ref,
        "step_gen": step_gen,
    }

In [18]:
def evaluate_toy(
    ref_dir,
    ref_stats_path
):
    # Load
    ref_universe = load_processed_universe(ref_dir)

    ref_stats_dict = load_stats_dict(ref_stats_path)

    # Build long panels
    panel_ref = make_long_panel(ref_universe, ref_stats_dict)

    # Global summaries
    global_ref = summarize_global_distribution(panel_ref)

    # Pathwise summaries
    path_ref = compute_pathwise_stats(ref_universe, ref_stats_dict)

    # Time summaries
    step_ref = stepwise_summary(panel_ref)

    # Print
    print("GLOBAL REFERENCE SUMMARY")
    display(global_ref)

    return {
        "panel_ref": panel_ref,
        "global_ref": global_ref,
        "path_ref": path_ref,
        "step_ref": step_ref,
    }

# Evaluation

In [19]:
results = evaluate_toy(ref_dir=ref_dir, ref_stats_path=ref_stats_path)
print(results)

GLOBAL REFERENCE SUMMARY


,mean,std,skew,kurtosis_excess,q01,q05,median,q95,q99
Open,0.000037,0.009225,-0.006217,1.252586,-0.023988,-0.015276,0.000049,0.015318,0.023994
High,0.014406,0.015618,1.055500,2.049867,-0.014338,-0.005694,0.011544,0.044197,0.063379
Low,-0.014197,0.015610,-1.065088,2.049736,-0.063231,-0.043992,-0.011300,0.005839,0.014299
Close,0.000160,0.018461,-0.008836,1.256982,-0.047992,-0.030516,0.000205,0.030707,0.048132
Volume,0.000402,0.211890,0.037607,0.028382,-0.487574,-0.345868,-0.000881,0.350763,0.500422


{'panel_ref':               Date      Open      High       Low     Close    Volume  \
0       1986-01-02 -0.016430 -0.008859 -0.016684 -0.013646 -0.083195   
1       1986-01-03  0.011090  0.029075 -0.005974  0.028075  0.269155   
2       1986-01-06 -0.010843  0.011720 -0.024141 -0.002795  0.008605   
3       1986-01-07 -0.002335  0.016171 -0.019562 -0.017223  0.019315   
4       1986-01-08 -0.004445  0.012708 -0.010693  0.007176  0.130480   
...            ...       ...       ...       ...       ...       ...   
2015995 2024-08-15 -0.020122  0.005045 -0.020370 -0.003225 -0.513521   
2015996 2024-08-16  0.015724  0.037631  0.008777  0.016358  0.076706   
2015997 2024-08-19  0.005217  0.012779 -0.025829 -0.012940  0.154496   
2015998 2024-08-20 -0.012047  0.002521 -0.013580 -0.003228  0.041828   
2015999 2024-08-21 -0.008098  0.014648 -0.010576 -0.008584 -0.110430   

            Ticker   Step  
0        FAKE_0001      0  
1        FAKE_0001      1  
2        FAKE_0001      2  
3        

In [20]:
results = evaluate_universes(
    ref_dir=ref_dir,
    ref_stats_path=ref_stats_path,
    gen_dir=gen_dir,
    gen_stats_path=gen_stats_path,
)

FileNotFoundError: [Errno 2] No such file or directory: '../data/model_generated_gbm/fake_stats_generated.pkl'